TITRE

INTRODUCTION

SOMMAIRE

Installation

In [ ]:
!pip install -r requirements.txt
#Modules:
import geopandas as gpd
import pandas as pd
#Fonctions:
import src.clear_data as cd
import src.get_data as gd

Préparation des données

Adresses

In [ ]:
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

Import et modifications/nettoyage des données

In [ ]:
#Import
df_dvf_raw = gd.fetch_dvf_api()
gdf_metro_raw = gd.fetch_metro_api()
#Nettoyage
gdf_metro = cd.clean_metro_data(gdf_metro_raw)
gdf_dvf = cd.clean_dvf_data(df_dvf_raw)
#Base fusionnée
gdf_final = cd.merge_dvf_by_line(gdf_dvf, gdf_metro)
gdf_final['prix_m2'] = gdf_final['valeur_fonciere'] / gdf_final['surface_reelle_bati']
print(gdf_final.head)
print(gdf_final.columns)

In [ ]:
import geopandas as gpd
import pandas as pd

Importation des deux bases de données :
- Demandes de valeurs foncières
- Topologie des points d'arrêt de métro du réseau STAR

Points d'arrêt de métro 

In [ ]:
# URL directe vers l'export GeoJSON du portail Open Data
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

# Lecture directe depuis l'URL
gdf_metro = gpd.read_file(url_metro)

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(gdf_metro.head())
print(gdf_metro.crs)

In [ ]:
print(gdf_metro.columns)


Demandes de Valeurs Foncières (DVF)

In [ ]:
# Exemple pour l'année 2023 (ou la plus récente disponible sur l'API DVF)
# URL des fichiers consolidés par Etalab
url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

# On précise le séparateur (souvent la virgule) et on gère la compression .gz
# précisez 'low_memory=False' pour éviter les warnings sur les types
df_dvf = pd.read_csv(url_dvf, compression='gzip', sep=',', low_memory=False)

# Pour filtrer sur Rennes (code commune 35238)
df_rennes = df_dvf[df_dvf['code_commune'] == '35238'].copy()

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(df_rennes.head())

In [ ]:
print(df_rennes.columns)

Modèle de prédiction du prix :

In [ ]:
from src.model import preparer_et_entrainer

# Supposons que df_rennes et gdf_metro sont déjà chargés
model_final, features_utilisees = preparer_et_entrainer(df_rennes, gdf_metro)

print("Modèle entraîné avec succès !")
print(f"Features utilisées : {features_utilisees}")

# Vous pouvez maintenant utiliser model_final pour faire des prédictions
# exemple_data = ... 
# prediction = model_final.predict(exemple_data)

In [ ]:

from src.model import preparer_et_entrainer, predire_impact_nouvelle_station

# Exemple : un appartement de 50m², 3 pièces, en 2026
surface = 50
pieces = 3
annee = 2026
code_type = 1 # Supposons que 1 soit l'appartement

# Scénario 1 : Le métro est à 2 km (2000m)
prix_actuel = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=2000)

# Scénario 2 : Le métro est à 0 m (nouvelle station en bas de l'immeuble)
prix_avec_metro = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=0)

print(f"Prix estimé actuel : {prix_actuel:.2f} €")
print(f"Prix estimé avec nouvelle station : {prix_avec_metro:.2f} €")
print(f"Plus-value théorique : {prix_avec_metro - prix_actuel:.2f} €")